# Traffic Multi-Violation YOLO11 GPU Retraining Notebook

This Google Colab notebook retrains the Traffic Multi-Violation detector on the **FULL `traffic dataset.v4i.yolov11` dataset** (5,238 train, 219 val, 218 test images) at inference size (`imgsz=640`) on a free cloud GPU (T4 / A100 / L4).

### Why Train on Google Colab GPU?
- **Local Host Machine**: Uses integrated **Intel Iris Xe Graphics** without an NVIDIA CUDA GPU.
- **Training Time**: On CPU, 60 epochs would take ~14 days (340 hours). On Google Colab free T4 GPU, 60 epochs takes **~25-30 minutes**.
- **Expected Outcome**: Significantly higher mAP across all 4 classes (`helmet`, `no helmet`, `number plate`, `triple riding`), eliminating rider-level misclassifications and plate collapse.

### Instructions
1. **Enable GPU**: Click **Runtime** -> **Change runtime type** -> select **T4 GPU** -> click **Save**.
2. **Upload Dataset**: Zip your `traffic dataset.v4i.yolov11` folder and upload it, or mount your Google Drive.
3. **Run all cells**: Click **Runtime** -> **Run all**.
4. **Download `best.pt`**: The last cell automatically downloads the trained `best.pt` to drop into `models/best.pt` in your local project.

In [ ]:
# Step 1: Verify NVIDIA GPU & CUDA Acceleration
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Step 2: Install Ultralytics and dependencies
!pip install -q ultralytics torchvision pyyaml
from ultralytics import YOLO
import os, sys, yaml, shutil
print("Ultralytics YOLO installed successfully.")

In [ ]:
# Step 3: Dataset Setup
# Choose Option A (Direct ZIP upload) or Option B (Google Drive)

DATASET_DIR = "/content/traffic_dataset"

# Option A: Unzip if you uploaded 'traffic dataset.v4i.yolov11.zip'
if os.path.exists("/content/traffic dataset.v4i.yolov11.zip"):
    !unzip -q "/content/traffic dataset.v4i.yolov11.zip" -d /content/traffic_dataset
elif os.path.exists("/content/traffic_dataset.zip"):
    !unzip -q /content/traffic_dataset.zip -d /content/traffic_dataset
elif os.path.exists("/content/traffic dataset.v4i.yolov11"):
    DATASET_DIR = "/content/traffic dataset.v4i.yolov11"
else:
    print("Please upload your dataset zip file to /content or mount Google Drive.")

print(f"Using dataset directory: {DATASET_DIR}")
if os.path.exists(DATASET_DIR):
    print("Train images:", len(os.listdir(os.path.join(DATASET_DIR, 'train', 'images'))))
    print("Val images:  ", len(os.listdir(os.path.join(DATASET_DIR, 'valid', 'images'))))
    print("Test images: ", len(os.listdir(os.path.join(DATASET_DIR, 'test', 'images'))))

In [ ]:
# Step 4: Write training data.yaml configuration
CLASS_NAMES = ['helmet', 'no helmet', 'number plate', 'triple riding']

data_yaml_dict = {
    'path': DATASET_DIR,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {i: n for i, n in enumerate(CLASS_NAMES)}
}

yaml_path = "/content/traffic_violation_full.yaml"
with open(yaml_path, 'w') as f:
    yaml.safe_dump(data_yaml_dict, f, sort_keys=False)

print("Generated YAML config:")
!cat /content/traffic_violation_full.yaml

In [ ]:
# Step 5: Retrain Detector with YOLO11s on GPU (60 Epochs, imgsz=640)
MODEL_NAME = 'yolo11s.pt'  # or 'yolo11n.pt' for even faster training
EPOCHS = 60
PATIENCE = 15
IMGSZ = 640
BATCH = 16

print(f"Starting full training run: model={MODEL_NAME}, epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH} on GPU...")
model = YOLO(MODEL_NAME)
results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=4,
    project='/content/runs/train_traffic_violation',
    name='full_dataset',
    exist_ok=True,
    plots=True
)

best_weights = '/content/runs/train_traffic_violation/full_dataset/weights/best.pt'
print(f"Training complete! Best weights saved to: {best_weights}")

In [ ]:
# Step 6: Validate and print per-class metrics on 'val' and 'test' splits
best_weights = '/content/runs/train_traffic_violation/full_dataset/weights/best.pt'
best_model = YOLO(best_weights)

for split in ('val', 'test'):
    r = best_model.val(data=yaml_path, split=split, imgsz=IMGSZ, device=0, plots=False, verbose=False)
    print(f"\n[EVALUATION] Split: {split.upper()}")
    print(f"  Overall mAP@0.5:     {r.box.map50:.3f}")
    print(f"  Overall mAP@0.5:0.95: {r.box.map:.3f}")
    for i, name in best_model.names.items():
        print(f"    {name:15s} Precision={r.box.p[i]:.3f} Recall={r.box.r[i]:.3f} mAP@0.5={r.box.ap50[i]:.3f}")

In [ ]:
# Step 7: Download best.pt to your computer
from google.colab import files
files.download('/content/runs/train_traffic_violation/full_dataset/weights/best.pt')
print("Downloaded best.pt. Copy this file to 'models/best.pt' in your project repository.")